In [33]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from scipy.stats import uniform, loguniform, randint

In [7]:
data = pd.read_csv('../datasets/Teen_Mental_Health_Dataset.csv')
data.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


In [27]:
categorical_features = ['gender', 'platform_usage', 'social_interaction_level']
numerical_features = data.drop(['depression_label'], axis=1).select_dtypes(include=np.number).columns

# data
X = data.drop(['depression_label'], axis=1)
y = data['depression_label']

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

# pipeline
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
])

# model
model = pipeline.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

In [28]:
target_names = ['not depressed', 'depressed']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")


MODEL EVALUATION
Training Accuracy: 98.02%
Test Accuracy: 97.50%
Overfitting Gap: 0.0052
✅ NO OVERFITTING (gap < 1%)
✅ No underfitting detected

Classification Report:
               precision    recall  f1-score   support

not depressed       0.97      1.00      0.99       234
    depressed       0.00      0.00      0.00         6

     accuracy                           0.97       240
    macro avg       0.49      0.50      0.49       240
 weighted avg       0.95      0.97      0.96       240


Confusion Matrix:
[[234   0]
 [  6   0]]
Correct predictions: 234
Total predictions: 240
Accuracy: 0.9750
Accuracy: 97.50%

not depressed Accuracy: 1.0000 (100.00%)
depressed Accuracy: 0.0000 (0.00%)

CROSS-VALIDATION
CV Scores: [0.97395833 0.97395833 0.97395833 0.97395833 0.97395833]
CV Mean: 0.9740
CV Std: 0.0000


C:\Users\rah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\rah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\rah\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [36]:
categorical_features = ['gender', 'platform_usage', 'social_interaction_level']
numerical_features = data.drop(['depression_label'], axis=1).select_dtypes(include=np.number).columns

# data
X = data.drop(['depression_label'], axis=1)
y = data['depression_label']

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

# pipeline
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, random_state=42))
])

# modeling using hyperparameter tuninig
param_distributions = {
    'classifier__C': loguniform(0.01, 100),  # Log scale for C
    'classifier__gamma': loguniform(0.001, 10),  # Log scale for gamma
    'classifier__kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'classifier__degree': [2, 3, 4],  # For poly kernel
    'classifier__coef0': uniform(0, 1),  # For poly/sigmoid
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,  # Number of parameter combinations to try
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1,
    random_state=42,
    return_train_score=True
)

random_search.fit(X_train, y_train)

model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

print("Best Parameters:")
print("=" * 50)
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Accuracy: {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters:
  classifier__C: 72.72394089120162
  classifier__coef0: 0.5582934536070976
  classifier__degree: 4
  classifier__gamma: 0.005686282517103598
  classifier__kernel: rbf

Best CV Accuracy: 0.9854


In [37]:
target_names = ['not depressed', 'depressed']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")



MODEL EVALUATION
Training Accuracy: 99.17%
Test Accuracy: 98.33%
Overfitting Gap: 0.0083
✅ NO OVERFITTING (gap < 1%)
✅ No underfitting detected

Classification Report:
               precision    recall  f1-score   support

not depressed       0.99      1.00      0.99       234
    depressed       0.75      0.50      0.60         6

     accuracy                           0.98       240
    macro avg       0.87      0.75      0.80       240
 weighted avg       0.98      0.98      0.98       240


Confusion Matrix:
[[233   1]
 [  3   3]]
Correct predictions: 236
Total predictions: 240
Accuracy: 0.9833
Accuracy: 98.33%

not depressed Accuracy: 0.9957 (99.57%)
depressed Accuracy: 0.5000 (50.00%)

CROSS-VALIDATION
CV Scores: [0.97395833 0.97395833 0.97395833 0.97395833 0.97395833]
CV Mean: 0.9740
CV Std: 0.0000
